# End-to-End Integration Demo

This notebook demonstrates the complete workflow from reading account information to building transactions.

In [1]:
import sys
sys.path.append('../00_setup')
from helpers import call_api, check_health, format_algo, short_address
from config import READER_URL, WRITER_URL, ALICE_ADDRESS, BOB_ADDRESS, ONE_ALGO

## Service Health Check

First, let's check the status of all services:

In [2]:
print("🔍 Checking Service Health")
print("=" * 30)

# Check Reader service
reader_health = await check_health(READER_URL)
print(f"📖 Reader MCP: {'✅ Healthy' if reader_health else '❌ Not available'}")

# Check Writer service
writer_health = await check_health(WRITER_URL)
print(f"✏️  Writer MCP: {'✅ Healthy' if writer_health else '❌ Not available'}")

if not reader_health or not writer_health:
    print("\n💡 To start services:")
    if not writer_health:
        print("   - Run: node notebooks/demo-writer-service.js")
    print("   - Or set USE_MOCK_MODE=True in config.py")

🔍 Checking Service Health
📖 Reader MCP: ✅ Healthy
✏️  Writer MCP: ✅ Healthy


## Step 1: Check Account Balances

Before making any transactions, let's check the current balances:

In [3]:
print("💰 Account Balance Check")
print("=" * 30)

# Get Alice's balance
alice_result = await call_api(f"{READER_URL}/tools/get_account_info", {"address": ALICE_ADDRESS})
if alice_result.get("success"):
    alice_balance = alice_result["account"]["amount"]
    print(f"👩 Alice: {format_algo(alice_balance)} ALGO")
else:
    alice_balance = 0
    print(f"👩 Alice: ❌ {alice_result.get('error')}")

# Get Bob's balance
bob_result = await call_api(f"{READER_URL}/tools/get_account_info", {"address": BOB_ADDRESS})
if bob_result.get("success"):
    bob_balance = bob_result["account"]["amount"]
    print(f"👨 Bob: {format_algo(bob_balance)} ALGO")
else:
    bob_balance = 0
    print(f"👨 Bob: ❌ {bob_result.get('error')}")

print(f"\n📊 Total: {format_algo(alice_balance + bob_balance)} ALGO")

💰 Account Balance Check
👩 Alice: 703.754899 ALGO
👨 Bob: 138105104.975914 ALGO

📊 Total: 138105808.730813 ALGO


## Step 2: Build a Transaction

Now let's build a payment transaction from Alice to Bob:

In [4]:
import datetime

print("🔨 Building Payment Transaction")
print("=" * 35)

# Transaction details
amount = ONE_ALGO  # 1 ALGO
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
note = f"End-to-end demo payment - {timestamp}"

transaction_data = {
    "fromAddress": ALICE_ADDRESS,
    "toAddress": BOB_ADDRESS,
    "microAlgos": amount,
    "note": note
}

print(f"💰 Amount: {format_algo(amount)} ALGO")
print(f"👩 From: {short_address(ALICE_ADDRESS)}")
print(f"👨 To: {short_address(BOB_ADDRESS)}")
print(f"📝 Note: {note}")

# Build the transaction
tx_result = await call_api(f"{WRITER_URL}/tools/build_payment_transaction", transaction_data)

if tx_result.get("success"):
    print("\n✅ Transaction Built Successfully!")
    print(f"🆔 Transaction ID: {tx_result.get('txId')}")
    print(f"💸 Network Fee: {tx_result.get('fee', 0)} microAlgos")
    print(f"🔢 Valid Rounds: {tx_result.get('firstRound')} - {tx_result.get('lastRound')}")
    
    # Store transaction info for next steps
    built_transaction = tx_result
else:
    print(f"\n❌ Transaction Build Failed: {tx_result.get('error')}")
    built_transaction = None

🔨 Building Payment Transaction
💰 Amount: 1.0 ALGO
👩 From: 7ZUECA7H...THAIOF6Q
👨 To: GD64YIY3...CDBBHU5A
📝 Note: End-to-end demo payment - 2025-09-19 23:04:52

✅ Transaction Built Successfully!
🆔 Transaction ID: HH6OARTHVAKY7AFFEHXOY2DMDCPMRU3NUIJJSJ724AI3UYACA7JQ
💸 Network Fee: 1000 microAlgos
🔢 Valid Rounds: 55716991 - 55717991


## Step 3: Transaction Analysis

Let's analyze the built transaction:

In [5]:
if built_transaction:
    print("🔍 Transaction Analysis")
    print("=" * 25)
    
    # Calculate total cost (amount + fee)
    total_cost = amount + built_transaction.get('fee', 1000)
    print(f"💰 Payment Amount: {format_algo(amount)} ALGO")
    print(f"💸 Network Fee: {format_algo(built_transaction.get('fee', 1000))} ALGO")
    print(f"💵 Total Cost: {format_algo(total_cost)} ALGO")
    
    # Check if Alice has sufficient balance
    if alice_balance >= total_cost:
        print(f"✅ Alice has sufficient balance ({format_algo(alice_balance)} ALGO)")
        remaining_balance = alice_balance - total_cost
        print(f"📊 Remaining after transaction: {format_algo(remaining_balance)} ALGO")
    else:
        print(f"❌ Insufficient balance! Needs {format_algo(total_cost - alice_balance)} more ALGO")
    
    # Show transaction validity window
    first_round = built_transaction.get('firstRound', 0)
    last_round = built_transaction.get('lastRound', 0)
    round_window = last_round - first_round
    print(f"⏰ Transaction valid for {round_window} rounds (~{round_window * 4.5:.1f} seconds)")
else:
    print("❌ No transaction to analyze")

🔍 Transaction Analysis
💰 Payment Amount: 1.0 ALGO
💸 Network Fee: 0.001 ALGO
💵 Total Cost: 1.001 ALGO
✅ Alice has sufficient balance (703.754899 ALGO)
📊 Remaining after transaction: 702.753899 ALGO
⏰ Transaction valid for 1000 rounds (~4500.0 seconds)


## Step 4: Simulate Post-Transaction State

Let's calculate what the balances would be after the transaction:

In [6]:
if built_transaction and alice_balance > 0 and bob_balance >= 0:
    print("🔮 Post-Transaction Simulation")
    print("=" * 35)
    
    # Calculate new balances
    transaction_fee = built_transaction.get('fee', 1000)
    alice_new_balance = alice_balance - amount - transaction_fee
    bob_new_balance = bob_balance + amount
    
    print("📊 Balance Changes:")
    print(f"👩 Alice: {format_algo(alice_balance)} → {format_algo(alice_new_balance)} ALGO")
    print(f"👨 Bob: {format_algo(bob_balance)} → {format_algo(bob_new_balance)} ALGO")
    
    # Show the transfer
    alice_change = alice_new_balance - alice_balance
    bob_change = bob_new_balance - bob_balance
    
    print("\n📈 Net Changes:")
    print(f"👩 Alice: {alice_change:+,} microAlgos ({format_algo(alice_change):+.6f} ALGO)")
    print(f"👨 Bob: {bob_change:+,} microAlgos ({format_algo(bob_change):+.6f} ALGO)")
    print(f"💸 Network Fee: -{transaction_fee} microAlgos (-{format_algo(transaction_fee):.6f} ALGO)")
    
    # Verify conservation (total should decrease by fee)
    total_change = alice_change + bob_change
    print(f"\n🧮 Total Change: {total_change} microAlgos (should equal -{transaction_fee})")
    
    if total_change == -transaction_fee:
        print("✅ Conservation verified: Only network fee is consumed")
    else:
        print("⚠️  Conservation check failed")
else:
    print("❌ Cannot simulate: Missing transaction or balance data")

🔮 Post-Transaction Simulation
📊 Balance Changes:
👩 Alice: 703.754899 → 702.753899 ALGO
👨 Bob: 138105104.975914 → 138105105.975914 ALGO

📈 Net Changes:
👩 Alice: -1,001,000 microAlgos (-1.001000 ALGO)
👨 Bob: +1,000,000 microAlgos (+1.000000 ALGO)
💸 Network Fee: -1000 microAlgos (-0.001000 ALGO)

🧮 Total Change: -1000 microAlgos (should equal -1000)
✅ Conservation verified: Only network fee is consumed


## Step 5: Integration Summary

Let's summarize what we accomplished in this end-to-end demo:

In [7]:
print("📋 End-to-End Integration Summary")
print("=" * 40)

steps_completed = []

# Check which steps were completed
if reader_health:
    steps_completed.append("✅ Connected to Reader MCP service")
else:
    steps_completed.append("❌ Reader MCP service unavailable")

if writer_health:
    steps_completed.append("✅ Connected to Writer MCP service")
else:
    steps_completed.append("❌ Writer MCP service unavailable")

if alice_result.get("success") and bob_result.get("success"):
    steps_completed.append("✅ Retrieved account balances")
else:
    steps_completed.append("❌ Failed to retrieve account balances")

if built_transaction:
    steps_completed.append("✅ Built payment transaction")
    steps_completed.append("✅ Analyzed transaction details")
    steps_completed.append("✅ Simulated post-transaction state")
else:
    steps_completed.append("❌ Failed to build transaction")

for step in steps_completed:
    print(step)

print("\n🎯 Integration Features Demonstrated:")
print("   • MCP service health checking")
print("   • Cross-service data flow (Reader → Writer)")
print("   • Real-time balance verification")
print("   • Transaction cost analysis")
print("   • State change simulation")
print("   • Error handling and fallbacks")

if built_transaction:
    print(f"\n🚀 Ready for Next Step: Sign and submit transaction {built_transaction.get('txId')}")
else:
    print("\n💡 Fix service connectivity issues and retry")

📋 End-to-End Integration Summary
✅ Connected to Reader MCP service
✅ Connected to Writer MCP service
✅ Retrieved account balances
✅ Built payment transaction
✅ Analyzed transaction details
✅ Simulated post-transaction state

🎯 Integration Features Demonstrated:
   • MCP service health checking
   • Cross-service data flow (Reader → Writer)
   • Real-time balance verification
   • Transaction cost analysis
   • State change simulation
   • Error handling and fallbacks

🚀 Ready for Next Step: Sign and submit transaction HH6OARTHVAKY7AFFEHXOY2DMDCPMRU3NUIJJSJ724AI3UYACA7JQ
